### Transformer Encoder 파트만 떼어내어 수백층으로 쌓아올린 모델
- 문장전체를 병렬로 한번에 입력받아서 앞뒤 양방향 의 문맥
### 2가지 사전학습 전략
- MLM(Masked Language Model) - 빈칸 맞추기
    - 문장 중간에 단어 15%를 무작위로 [MASK]라는 빈칸으로 가려
    - 모델은 주변의 남은 단어로 학습( context를 파악) - 문맥파알
- NSP(Next Sentence Prediction) - 다음문장 맞추기
    - 두개의 문장 A와 B를 연속으로 보여줌
    - B가 A에 연결되는 진짜인지 학습(두 문장사이의 논리적 연결성)
### 서브워드(subword)토크나이져와 Special Token
- 자연어 처리에서 골칫거리 사전에 없는 신조어(OOV:Out-of-Vopcabulary) 가 입력으로 들어오면 에러
- Bert는 이 상황을 WordPiece라는 기술로 해결
    - WordPiece 토크나이져 : ~~ 앵그리 빡빡... 액그리, 빡 빡 쵀대한 조각내서 의미를 분석
    - Special Tokens : 
        - [CLS](Classification) : 문장의 맨 앞에붙는 토큰, 문장 전체의 핵심 요약정보를 담는다
        - [SEP](Separator) : 문장과 문장을 구별, 문장이 끝남을 알려주는 마침표
        - [MASK] : 빈칸채우기에 사용되는 마스크

In [1]:
from transformers import BertTokenizer
tokenizer_en = BertTokenizer.from_pretrained('bert-base-uncased')
text_en = 'I am learning natual language processing'
raw_tokens =  tokenizer_en.tokenize(text_en)
print(raw_tokens)

# 모델에 실제로 들어가기전 앞뒤 특수토큰
encoded_dict = tokenizer_en(text_en,add_special_tokens=True)
input_ids = encoded_dict['input_ids']
print(f'숫자로 변환된 결과 : {input_ids}')
# 숫자로 변환된 id를 다시 문자로 디코딩 확인
restored_tokens = tokenizer_en.convert_ids_to_tokens(input_ids)
print(f'숫자를 다시 글자로 복원한 결과 : {restored_tokens}')

['i', 'am', 'learning', 'nat', '##ual', 'language', 'processing']
숫자로 변환된 결과 : [101, 1045, 2572, 4083, 14085, 8787, 2653, 6364, 102]
숫자를 다시 글자로 복원한 결과 : ['[CLS]', 'i', 'am', 'learning', 'nat', '##ual', 'language', 'processing', '[SEP]']


### 입력 텐서의 3대장(input ids, Attention Mask, Token Type ids)
- attention mask가 필요한 이유 : 행렬연산을위해 길이를 맞추면 짧은 문장은 pad 토큰(0)으로 채운다
- Attention 연산과정에서 쓸모없는 0을 의미가 있는 줄 알고 학습, 0 1로 마스크를 씌워서 필요없는 부분은 학습배제

In [ ]:
sentences = [
    "I love NLP.",
    "Transformers are incredibly powerful and fast."
]
batch_inputs =  tokenizer_en(sentences,padding=True, return_tensors='pt')
for k,v in batch_inputs.items():
    print(k,v)
# token_type_ids  문장 A는 0, B는 1 --> 지금 문장이하나... 그래서 다 0    

input_ids tensor([[  101,  1045,  2293, 17953,  2361,  1012,   102,     0,     0],
        [  101, 19081,  2024, 11757,  3928,  1998,  3435,  1012,   102]])
token_type_ids tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0]])
attention_mask tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [ ]:
# 카카오등에서 사전학습한 한국어 전용 BERT 토크나이져
tokenized_ko = BertTokenizer.from_pretrained('klue/bert-base')
# 신조어가포함된 한국어 영화리뷰 텍스트
real_korean_review = '이 영화는 정말 도무지 설명할수가 없어요..킹 왕짱 ㅋㅋㅋ'
ko_tokens = tokenized_ko.tokenize(real_korean_review)
ko_input_ids = tokenized_ko(real_korean_review)['input_ids']
ko_resotred =  tokenized_ko.convert_ids_to_tokens(ko_input_ids)
print(ko_resotred)
# #기호의 의미  나는 독립된 단어가 아니라 앞 글자 공백없이 바로 붙어있는 단어라는 문법적 단서를 모델에 제시

['[CLS]', '이', '영화', '##는', '정말', '도무지', '설명', '##할', '##수', '##가', '없', '##어요', '.', '.', '킹', '왕', '##짱', 'ㅋㅋㅋ', '[SEP]']
